# CS 195: Natural Language Processing
## PyTorch Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F4_2_PyTorchEmbeddings.ipynb)


## References

[SLP: Embeddings, Chapter 5 of Speech and Language Processing by Daniel Jurafsky & James H. Martin](https://web.stanford.edu/~jurafsky/slp3/5.pdf)

[Word2Vec Tutorial - The Skip-Gram Model by Chris McCormick](http://mccormickml.com/2016/04/19/word2vec-tutorial-the-skip-gram-model/)

[Word2Vec - Negative Sampling made easy by Munesh Lakhey](https://medium.com/@mnshonco/word2vec-negative-sampling-made-easy-9a587cb4695f)

[PyTorch `nn.Embedding` documentation](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

[PyTorch `torch.nn.functional.one_hot` documentation](https://pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)


In [ ]:
#import sys
#!{sys.executable} -m pip install datasets torch scikit-learn transformers tokenizers

## Reorganization of where we left off from last time

In [2]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
import random 
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim

def train_tokenizer(sentences, vocabulary_size=200):
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(vocab_size=vocabulary_size)

    tokenizer.train_from_iterator(sentences, trainer)
    return tokenizer


def make_skipgrams(sequence, vocabulary_size, window_size=3):
    couples = []
    labels = []

    for i in range(len(sequence)):
        target = sequence[i]
        left = max(0, i - window_size)
        right = min(len(sequence), i + window_size + 1)

        for j in range(left, right):
            if i != j:
                context = sequence[j]

                # positive pair
                couples.append((target, context))
                labels.append(1)

                # generate a random negative pair (in real life, you might want to generate several negative pairs per positive pair)
                negative_context = random.randint(1, vocabulary_size - 1)
                # TODO: we should probably check to make sure this isn't actually a positive pair
                couples.append((target, negative_context))
                labels.append(0)

    return [couples, labels]

def make_skipgrams_batch(tokenized_sentences, vocabulary_size, window_size=3):
    all_couples = []
    all_labels = []
    for tokenized_sentence in tokenized_sentences:
        couples, labels = make_skipgrams(tokenized_sentence, vocabulary_size, window_size)
        all_couples.extend(couples)
        all_labels.extend(labels)

    return [all_couples, all_labels]


def prepare_one_hot_inputs(couples, labels, vocabulary_size):
    inputs = []
    for target_word, context_word in couples:
        target_one_hot = F.one_hot(torch.tensor(target_word), num_classes=vocabulary_size)
        context_one_hot = F.one_hot(torch.tensor(context_word), num_classes=vocabulary_size)
        inputs.append(torch.cat([target_one_hot, context_one_hot]))

    inputs_array = torch.stack(inputs).float()
    labels_array = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    return inputs_array, labels_array


def train_embedding_model(inputs_array, labels_array, vocabulary_size):

    embedding_model = nn.Sequential(
        nn.Linear(vocabulary_size * 2, 50),
        nn.ReLU(),
        nn.Linear(50, 1)
    )

    loss_fn = nn.BCEWithLogitsLoss() # don't forget - this includes the sigmoid squashing function 
    optimizer = optim.Adam(embedding_model.parameters(), lr=0.0001)

    for epoch in range(20000):
        logits = embedding_model(inputs_array)
        loss = loss_fn(logits, labels_array)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 1000 == 0:
            print(f"epoch {epoch+1}, loss={loss.item():.4f}")

    return embedding_model

def get_embedding(word, tokenizer, embedding_model):
    word_ids = tokenizer.encode(word).ids
    first_layer_weights = embedding_model[0].weight.detach()
    return first_layer_weights[:, word_ids[0]] # use first subword token for this demo



In [3]:
sentences = [
    "I adopted some dogs from the animal shelter",
    "don't you know that dogs and cats both like scritches",
    "are cats or dogs your favorite animal",
    "I have heard that dogs can be obedient",
    "I have heard that cats can be independent",
    "sharks live in the ocean",
    "many birds fly to get around",
    "dogs and cats are common household pets",
    "cats and dogs both need food and water",
    "my dog and my cat play together",
    "cats and dogs can live in the same home",
    "the puppy and kitten were adopted together",
    "fish swim in the ocean",
    "whales and sharks live in the ocean",
    "boats travel across the ocean",
    "the ocean water is deep and salty",
    "coral reefs are in the ocean"
]

tokenizer = train_tokenizer(sentences, vocabulary_size=200)

tokenized_sentences = []
for sentence in sentences:
    ids = tokenizer.encode(sentence).ids
    tokens = tokenizer.encode(sentence).tokens
    tokenized_sentences.append(ids)

print("Here's an example of some tokens:", tokens)

couples, labels = make_skipgrams_batch(tokenized_sentences, vocabulary_size=tokenizer.get_vocab_size(), window_size=2)

inputs_array, labels_array = prepare_one_hot_inputs(couples, labels, tokenizer.get_vocab_size())

embedding_model = train_embedding_model(inputs_array, labels_array, vocabulary_size=tokenizer.get_vocab_size())





Here's an example of some tokens: ['coral', 'reefs', 'are', 'in', 'the', 'ocean']
epoch 1000, loss=0.4167
epoch 2000, loss=0.2688
epoch 3000, loss=0.1973
epoch 4000, loss=0.1451
epoch 5000, loss=0.1045
epoch 6000, loss=0.0763
epoch 7000, loss=0.0591
epoch 8000, loss=0.0491
epoch 9000, loss=0.0433
epoch 10000, loss=0.0401
epoch 11000, loss=0.0382
epoch 12000, loss=0.0371
epoch 13000, loss=0.0364
epoch 14000, loss=0.0361
epoch 15000, loss=0.0358
epoch 16000, loss=0.0357
epoch 17000, loss=0.0356
epoch 18000, loss=0.0356
epoch 19000, loss=0.0356
epoch 20000, loss=0.0355


In [4]:

cats_embedding = get_embedding("cats", tokenizer, embedding_model)
dogs_embedding = get_embedding("dogs", tokenizer, embedding_model)
ocean_embedding = get_embedding("ocean", tokenizer, embedding_model)

print(cats_embedding)
print(dogs_embedding)

# distance between dogs and cats should be smaller than distance between dogs and ocean
print(torch.sum((dogs_embedding - cats_embedding) ** 2).item())
print(torch.sum((dogs_embedding - ocean_embedding) ** 2).item())

# let's also look at cosine similarity, which is a common way to measure similarity between vectors that ignores differences in magnitude
# a negative cosine similarity means the vectors are pointing in opposite directions, a positive cosine similarity means they are pointing in the same direction, and a cosine similarity close to 0 means they are orthogonal (i.e. not similar at all)
dogs_cats_similarity = F.cosine_similarity(dogs_embedding, cats_embedding, dim=0).item()
dogs_ocean_similarity = F.cosine_similarity(dogs_embedding, ocean_embedding, dim=0).item()
print("dogs, cats similarity", dogs_cats_similarity)
print("dogs, ocean similarity", dogs_ocean_similarity)


tensor([ 1.0034, -0.3097,  0.7628,  0.9048, -0.1995,  0.8928, -0.9590,  1.0422,
        -0.9632, -0.8163,  0.9871, -1.0050,  0.8518,  0.6025,  0.7179,  0.9731,
         0.9753, -0.5866,  0.9121,  0.8623,  0.8090, -0.0957,  0.4025,  0.9329,
         0.3348,  0.9783, -0.7822,  1.0392,  0.8051, -0.5617, -0.3530,  0.4534,
         1.0348, -0.9267,  0.4606,  0.6732,  0.9732,  0.9392,  0.1477, -0.7531,
         0.8065,  0.9593,  0.7510,  1.0513,  0.8332,  0.8668,  0.2704,  1.0064,
        -1.1115,  1.0548])
tensor([ 0.9912,  0.7549, -0.8876,  0.9752,  0.6758,  1.0060, -0.1377,  0.9492,
         1.1546,  0.1221,  0.5530, -1.1096,  0.9633,  0.9049, -0.6755, -0.1574,
         0.8158,  0.9135,  0.9035, -1.0275,  1.0812,  0.8540,  1.0357, -0.2111,
        -0.7379,  1.2132,  0.6543, -0.5897,  0.9394,  1.0768,  1.0368,  0.8157,
        -0.4199,  1.1521,  0.7990,  0.8852,  1.0131, -0.7933,  0.1477, -0.0101,
         0.1027,  1.2289,  0.8020,  0.9808,  0.4029,  0.6917, -0.2517, -0.2580,
        -0.97

## Dataset for today

AG News dataset
* short news articles
* four classes: World, Sports, Business, Sci/Tech

https://huggingface.co/datasets/fancyzhx/ag_news


In [5]:
from datasets import load_dataset
data = load_dataset("ag_news")

print(data["train"]["text"][0])

# 0 is World
# 1 is Sports
# 2 is Business
# 3 is Sci/Tech
print(data["train"]["label"][0])

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
2


## Group Exercise

Try creating word embeddings for the AG News dataset. *Note that you will only be able to do a very small subset with this code. Start with 10 examples and work your way up*

* How big of a vocabulary did you need to use to get reasonable tokens for this data?
* Once you use more examples, you'll start to notice that you're going to run out of memory and the kernel will crash. What do you think the issue is? Can you think of a way to do the same idea but keep less data in memory at any given time?



## The PyTorch Embedding Layer
PyTorch provides an `nn.Embedding` layer, which is a linear layer with a lookup table. It allows you to input a token id and look up a row vector (akin to the linear node associated with that id).

It's mathematically equivalent to the one-hot encoding + `nn.Linear` layer we saw before, but it much more memory efficient - we don't have to waste space storing all of those 0s. 

Let's try the same experiment as before but using the `nn.Embedding` layer. 


In [29]:
def train_embedding_model_v2(couples, labels, vocabulary_size):

    embedding_model = nn.Embedding(vocabulary_size, 50)
    skipgram_classifier = nn.Linear(2*50, 1) # target emb + context emb

    loss_fn = nn.BCEWithLogitsLoss() # don't forget - this includes the sigmoid squashing function 
    optimizer = optim.Adam(list(embedding_model.parameters()) + list(skipgram_classifier.parameters()), lr=0.0003) # concatenate the parameters for the embedding model and skipgram classifier

    pair_ids = torch.tensor(couples, dtype=torch.long)                # [N, 2]
    labels_tensor = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) # [N, 1]

    for epoch in range(5000):
        target_ids = pair_ids[:, 0] # [N]
        context_ids = pair_ids[:, 1] # [N]

        target_embeddings = embedding_model(target_ids)  # [N, 50]
        context_embeddings = embedding_model(context_ids)  # [N, 50]

        target_context_together = torch.cat([target_embeddings, context_embeddings], dim=1)  # [N, 100]

        logits = skipgram_classifier(target_context_together)
        loss = loss_fn(logits, labels_tensor )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 500 == 0:
            print(f"epoch {epoch+1}, loss={loss.item():.4f}")

    return embedding_model

def get_embedding_v2(word, tokenizer, embedding_model):
    word_ids = tokenizer.encode(word).ids[0] # use first subword token for this demo
    word_weights = embedding_model.weight.detach()
    return word_weights[word_ids]

sentences = [
    "I adopted some dogs from the animal shelter",
    "don't you know that dogs and cats both like scritches",
    "are cats or dogs your favorite animal",
    "I have heard that dogs can be obedient",
    "I have heard that cats can be independent",
    "sharks live in the ocean",
    "many birds fly to get around",
    "dogs and cats are common household pets",
    "cats and dogs both need food and water",
    "my dog and my cat play together",
    "cats and dogs can live in the same home",
    "the puppy and kitten were adopted together",
    "fish swim in the ocean",
    "whales and sharks live in the ocean",
    "boats travel across the ocean",
    "the ocean water is deep and salty",
    "coral reefs are in the ocean"
]

tokenizer = train_tokenizer(sentences, vocabulary_size=200)

tokenized_sentences = []
for sentence in sentences:
    ids = tokenizer.encode(sentence).ids
    tokens = tokenizer.encode(sentence).tokens
    tokenized_sentences.append(ids)

print(tokens)

couples, labels = make_skipgrams_batch(tokenized_sentences, vocabulary_size=tokenizer.get_vocab_size(), window_size=2)
embedding_model_v2 = train_embedding_model_v2(couples, labels, vocabulary_size=tokenizer.get_vocab_size())




['coral', 'reefs', 'are', 'in', 'the', 'ocean']
epoch 500, loss=0.4195
epoch 1000, loss=0.3230
epoch 1500, loss=0.3049
epoch 2000, loss=0.3001
epoch 2500, loss=0.2983
epoch 3000, loss=0.2975
epoch 3500, loss=0.2970
epoch 4000, loss=0.2967
epoch 4500, loss=0.2966
epoch 5000, loss=0.2965


In [30]:

cats_embedding = get_embedding_v2("cats", tokenizer, embedding_model_v2)
dogs_embedding = get_embedding_v2("dogs", tokenizer, embedding_model_v2)
ocean_embedding = get_embedding_v2("ocean", tokenizer, embedding_model_v2)

print(cats_embedding)
print(dogs_embedding)

# distance between dogs and cats should be smaller than distance between dogs and ocean
print(torch.sum((dogs_embedding - cats_embedding) ** 2).item())
print(torch.sum((dogs_embedding - ocean_embedding) ** 2).item())

# let's also look at cosine similarity, which is a common way to measure similarity between vectors that ignores differences in magnitude
# a negative cosine similarity means the vectors are pointing in opposite directions, a positive cosine similarity means they are pointing in the same direction, and a cosine similarity close to 0 means they are orthogonal (i.e. not similar at all)
dogs_cats_similarity = F.cosine_similarity(dogs_embedding, cats_embedding, dim=0).item()
dogs_ocean_similarity = F.cosine_similarity(dogs_embedding, ocean_embedding, dim=0).item()
print("dogs, cats similarity", dogs_cats_similarity)
print("dogs, ocean similarity", dogs_ocean_similarity)

tensor([-1.1894, -0.3011, -1.0305, -1.2189,  0.7533,  1.1182, -1.4845, -0.8735,
        -1.4293,  2.5282, -0.0406,  1.1639,  0.3438,  1.3515,  0.9803,  0.4391,
        -1.7663,  0.4266,  1.7434, -1.5325,  1.6689,  0.6913,  0.3718,  0.7833,
         2.3844, -1.6433, -0.1928, -0.1200, -0.9636, -0.3084, -0.3287,  0.6702,
         0.7434, -0.5538,  0.5809, -1.2299,  0.0690,  0.2024, -1.0377, -0.0476,
         1.7574,  0.1504,  0.3947,  0.9887,  1.9825, -0.6312,  0.3071, -0.2003,
        -0.4933,  0.0971])
tensor([-1.1153,  0.0182, -1.0883, -0.3306,  0.4938, -1.0888,  2.7576, -2.2433,
         0.1567, -0.1348,  1.2572,  1.2124,  1.4193, -1.4720,  2.9124,  0.8527,
        -0.1905,  0.6330, -1.2822,  0.8336, -0.0693, -1.8210, -0.5701, -0.2616,
         2.0807, -1.1239, -0.1739, -2.2497, -1.7614, -0.3880,  0.0184, -1.5142,
         1.2545, -1.0126,  0.1048, -0.3293, -1.3243,  0.2847,  0.2984,  0.3890,
         1.0662, -1.2351,  0.5707,  1.3407, -0.4611, -0.3479,  1.2202, -1.4390,
         0.90

### Caveat

With such a small dataset, we're not actually getting good embeddings yet. 

## Applied Exploration

Create word embeddings for a larger portion of the AG News dataset, say 5000 texts

Show some example word embeddings for some words that appear in the dataset (*cats* and *dogs* may not be good examples for this one)

Describe your results and reflect on them
* How big of a vocabulary did you need to use?
* What learning rate and number of training epochs do you think are appropriate? Why?
* How could you go about figuring out if these embeddings are useful?

## Adding an Embedding layer to your model for other learning tasks

First, let's prepare the data
* We could use the same kind of tokenizer, or just use an existing one - let's go back to doing it with a pretrained Hugging Face tokenizer


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

data = load_dataset("ag_news")

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]
train_labels = data["train"]["label"]
test_labels = data["test"]["label"]

hf_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
hf_tokenizer.add_special_tokens({'pad_token': '[PAD]'}) # for some reason this tokenizer doesn't have a pad token by default, but we need one to be able to batch our inputs together, so we'll just add a new special token for padding

tokenized_train_texts = hf_tokenizer(list(train_texts), truncation=True, padding=True, max_length=128, return_tensors="pt")
tokenized_test_texts = hf_tokenizer(list(test_texts), truncation=True, padding=True, max_length=128, return_tensors="pt")

X_train = tokenized_train_texts["input_ids"]
X_test = tokenized_test_texts["input_ids"]

y_train = torch.tensor(np.array(train_labels), dtype=torch.long)
y_test = torch.tensor(np.array(test_labels), dtype=torch.long)


## Model with an Embedding layer

We'll set up the embedding layer and sequential model as before

The optimizer needs to have a concatenation of the parameters for both parts

In [8]:
embedding = nn.Embedding(len(hf_tokenizer), 50) # len(hf_tokenizer) is the vocab size including the new padding token
classifier = nn.Sequential(
    nn.Linear(50, 100),
    nn.ReLU(),
    nn.Linear(100, 4)
)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(embedding.parameters()) + list(classifier.parameters()), lr=0.01)

## Training Loop with Embedding layer

The input goes into the embedding layer
* returns an embedding for each words, so we aggregate them with the *mean* to get an embedding for the whole training example

In [9]:


for epoch in range(100):
    optimizer.zero_grad()

    emb = embedding(X_train) # [batch_size, seq_len, embedding_dim]
    pooled_emb = emb.mean(dim=1) # simple way to get a single vector [batch_size, embedding_dim] for the whole sequence - just average the token embeddings
    logits = classifier(pooled_emb)
    loss = loss_fn(logits, y_train)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")



Epoch 1, loss = 1.3864
Epoch 11, loss = 1.3261
Epoch 21, loss = 1.0533
Epoch 31, loss = 0.6228
Epoch 41, loss = 0.3929
Epoch 51, loss = 0.2925
Epoch 61, loss = 0.2287
Epoch 71, loss = 0.1849
Epoch 81, loss = 0.1528
Epoch 91, loss = 0.1272


## Evaluating works similarly

In [10]:
with torch.no_grad():
    emb = embedding(X_test)
    pooled_emb = emb.mean(dim=1)
    logits = classifier(pooled_emb)
    predicted_labels = logits.argmax(dim=1)
    accuracy = (predicted_labels == y_test).float().mean()

print("Accuracy:", accuracy.item())

Accuracy: 0.902236819267273


## Applied Exploration

Perform a text classification experiment with another classification data set
* Try different embedding vector lengths (other than just 50 as we did here)
* Experiment with different neural network structures, learning rates, and number of epochs

Report your results and reflect on them
* Describe your dataset
* Describe what you did
* Report the results you observed
* Discuss any interesting insights